In [1]:
# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd

# 環境変数の取得
load_dotenv()

# OpenAI APIクライアントを生成
client = OpenAI(api_key=os.environ['API_KEY'])

# モデル名
MODEL_NAME = "gpt-4o-mini"

In [3]:
# 1. Excelファイルを読み込む
df = pd.read_excel('サンプルデータ.xlsx', sheet_name='売上データ')
# データフレームを表示して確認
df.head()

,カテゴリー,商品コード,商品名,売上日,単価,数量,原価
0,食品,1001,りんご,2023-01-01,200,50,120
1,食品,1002,バナナ,2023-01-01,150,100,80
2,食品,1003,牛乳,2023-01-02,180,80,100
3,衣服,2001,Tシャツ,2023-01-02,1500,20,800
4,衣服,2002,ジーンズ,2023-01-03,5000,10,2500


In [4]:
# 2. データをLLM用にテキスト形式に変換
# データフレーム全体を文字列に変換
sales_data_text = df.astype(str)
prompt_text = f"売上データ:\n{sales_data_text}\nこの売上データの傾向を分析してください。"
# 表示して確認
print(prompt_text)

売上データ:
    カテゴリー 商品コード      商品名         売上日    単価   数量    原価
0      食品  1001      りんご  2023-01-01   200   50   120
1      食品  1002      バナナ  2023-01-01   150  100    80
2      食品  1003       牛乳  2023-01-02   180   80   100
3      衣服  2001     Tシャツ  2023-01-02  1500   20   800
4      衣服  2002     ジーンズ  2023-01-03  5000   10  2500
..    ...   ...      ...         ...   ...  ...   ...
235    衣服  2077   レインパンツ  2023-04-28  2000   18  1000
236    食品  1085      ザクロ  2023-04-29   600   40   300
237   日用品  3077    バスブラシ  2023-04-29   400   60   200
238    衣服  2078  レインシューズ  2023-04-30  2500   15  1250
239    食品  1086    ココナッツ  2023-04-30   300   80   150

[240 rows x 7 columns]
この売上データの傾向を分析してください。


In [5]:
# 3. OpenAI APIの呼び出し

# 役割を設定
role = "あなたはマーケティング分野に精通したデータサイエンティストです。企業の成長をサポートするために、効果的なインサイトを提供します。"

# APIへリクエスト
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": role},
        {"role": "user", "content": prompt_text},
    ],
)

# LLMからの回答を表示
print(response.choices[0].message.content.strip())

売上データを分析するために、以下の視点からデータの傾向を考察します。

### 1. カテゴリー別の売上分析
- **食品**、**衣服**、**日用品**に分けられているため、それぞれのカテゴリーの総売上と数量を集計します。これにより、どのカテゴリーがより収益性が高いかを確認します。
  
  - 食品：売上高、数量
  - 衣服：売上高、数量
  - 日用品：売上高、数量

### 2. 売上の時間的傾向
- 日付に基づいて、売上の時間的な傾向を分析します。具体的には、月別、週別、あるいは日別の売上トレンドを把握します。特定の日や月に売上が急増または減少している場合、その原因を探ります。

### 3. 商品別のパフォーマンス
- 商品コードや商品名ごとに売上を集計し、最も人気のある商品や、逆に売上が落ちている商品を特定します。また、原価と単価から利益を計算し、利益率の高い商品も分析します。

### 4. 原価と利益の分析
- 各商品の原価と売上単価から、各商品の利益を計算し、総利益を求めます。利益率が高い商品や、コストを削減する必要がある商品を特定できます。

### 5. レバレッジポイントの特定
- どの商品の売上を伸ばすことで全体の売上や利益が改善されるか、さらなるマーケティング施策やプロモーションを行うべき商品を見つけます。

### 6. デモグラフィック分析
- 売上データが地域や顧客セグメントと関連している場合、その分析も行います。例えば、特定の地域で人気のある商品を特定することは、プロモーション活動に役立ちます。

### 結論
具体的な数値がないため一般的な分析手法を提案しましたが、実際の分析を行うためには、データを視覚化したり、詳細な数値を計算したりする必要があります。このような分析を通じて、より明確なビジネスインサイトを得ることができます。具体的な分析を行うためには、データの集計や視覚化ツール（Excel、PythonのPandasなど）を使うと良いでしょう。


In [6]:
# 4. 分析結果をデータフレームに変換
result_list = response.choices[0].message.content.strip().split("\n")
df_out = pd.DataFrame(result_list, columns=['結果'])
print(df_out)

                                                   結果
0                  売上データを分析するために、以下の視点からデータの傾向を考察します。
1                                                    
2                                  ### 1. カテゴリー別の売上分析
3   - **食品**、**衣服**、**日用品**に分けられているため、それぞれのカテゴリーの総...
4                                                    
5                                         - 食品：売上高、数量
6                                         - 衣服：売上高、数量
7                                        - 日用品：売上高、数量
8                                                    
9                                     ### 2. 売上の時間的傾向
10  - 日付に基づいて、売上の時間的な傾向を分析します。具体的には、月別、週別、あるいは日別の売...
11                                                   
12                                 ### 3. 商品別のパフォーマンス
13  - 商品コードや商品名ごとに売上を集計し、最も人気のある商品や、逆に売上が落ちている商品を特...
14                                                   
15                                    ### 4. 原価と利益の分析
16  - 各商品の原価と売上単価から、各商品の利益を計算し、総利益を求めます。利益率が高い商品や、...
17                          

In [7]:
# 5. 結果をExcelファイルに保存
df_out.to_excel("売上データ分析結果.xlsx", index=False)

In [8]:
# ワークフロー化
print("処理を開始します。")

# 1. Excelファイルを読み込む
df = pd.read_excel('サンプルデータ.xlsx', sheet_name='売上データ')

# 2. データをLLM用にテキスト形式に変換
sales_data_text = df.astype(str)
prompt_text = f"売上データ:\n{sales_data_text}\nこの売上データの傾向を分析してください。"

# 3. OpenAI APIの呼び出し
# 役割を設定
role = "あなたはマーケティング分野に精通したデータサイエンティストです。企業の成長をサポートするために、効果的なインサイトを提供します。"
# APIへリクエスト
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": role},
        {"role": "user", "content": prompt_text},
    ],
)

# 4. 分析結果をデータフレームに変換
result_list = response.choices[0].message.content.strip().split("\n")
df_out = pd.DataFrame(result_list, columns=['結果'])

# 5. 結果をExcelファイルに保存
df_out.to_excel("売上データ分析結果.xlsx", index=False)

print("Excelファイルに分析結果を保存しました。")

処理を開始します。
Excelファイルに分析結果を保存しました。
